QUE : 1 -->

Create a simple neural network using Keras to classify handwritten digits from the MNIST dataset, then experiment by changing the number of layers and neurons in your model to see how accuracy changes.Hint: Start with one hidden layer, then try adding another, and vary the number of neurons (e.g., 32, 64, 128).

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten

# Load MNIST dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Normalize pixel values
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0


# Function to create and train model
def create_model(neurons, layers):
    model = Sequential()

    # Input layer
    model.add(Flatten(input_shape=(28, 28)))

    # Add hidden layers
    for i in range(layers):
        model.add(Dense(neurons, activation='relu'))

    # Output layer
    model.add(Dense(10, activation='softmax'))

    # Compile model
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


# Experiment 1: One hidden layer with 32 neurons
model1 = create_model(32, 1)

history1 = model1.fit(
    x_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_data=(x_test, y_test)
)

accuracy1 = model1.evaluate(x_test, y_test)[1]


# Experiment 2: Two hidden layers with 64 neurons
model2 = create_model(64, 2)

history2 = model2.fit(
    x_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_data=(x_test, y_test)
)

accuracy2 = model2.evaluate(x_test, y_test)[1]


# Experiment 3: Two hidden layers with 128 neurons
model3 = create_model(128, 2)

history3 = model3.fit(
    x_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_data=(x_test, y_test)
)

accuracy3 = model3.evaluate(x_test, y_test)[1]


# Print results
print("Model 1 (1 Layer, 32 Neurons) Accuracy:", accuracy1)
print("Model 2 (2 Layers, 64 Neurons) Accuracy:", accuracy2)
print("Model 3 (2 Layers, 128 Neurons) Accuracy:", accuracy3)

QUE : 2 -->

Train your MNIST model using different batch sizes (e.g., 16, 32, 64) and epochs (e.g., 5, 10, 20), and record how the training time and validation accuracy change for each combination.

In [ ]:
import tensorflow as tf
import time

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten

# Load MNIST dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Normalize data
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0


# Function to create model
def create_model():
    model = Sequential([
        Flatten(input_shape=(28, 28)),
        Dense(128, activation='relu'),
        Dense(10, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


# Experiment settings
batch_sizes = [16, 32, 64]
epochs_list = [5, 10, 20]

results = []

# Train model with different batch sizes and epochs
for batch in batch_sizes:
    for epochs in epochs_list:

        print("\nTraining with Batch Size:", batch, 
              "and Epochs:", epochs)

        model = create_model()

        start_time = time.time()

        history = model.fit(
            x_train,
            y_train,
            epochs=epochs,
            batch_size=batch,
            validation_data=(x_test, y_test),
            verbose=0
        )

        end_time = time.time()

        training_time = end_time - start_time

        val_accuracy = history.history['val_accuracy'][-1]

        results.append([
            batch,
            epochs,
            training_time,
            val_accuracy
        ])


# Print results
print("\nBatch Size | Epochs | Training Time | Validation Accuracy")
print("-" * 60)

for result in results:
    print(
        result[0],
        "        |",
        result[1],
        "     |",
        round(result[2], 2),
        "seconds     |",
        round(result[3], 4)
    )

QUE : 3 -->

Modify your Keras model to use three different optimizers (SGD, Adam, RMSprop) and compare the results by plotting training and validation loss for each optimizer.

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten

# Load MNIST dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Normalize data
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0


# Function to create model
def create_model():
    model = Sequential([
        Flatten(input_shape=(28, 28)),
        Dense(128, activation='relu'),
        Dense(10, activation='softmax')
    ])
    return model


# Different optimizers
optimizers = {
    "SGD": tf.keras.optimizers.SGD(),
    "Adam": tf.keras.optimizers.Adam(),
    "RMSprop": tf.keras.optimizers.RMSprop()
}


history_results = {}


# Train model with each optimizer
for name, optimizer in optimizers.items():

    print("Training with", name)

    model = create_model()

    model.compile(
        optimizer=optimizer,
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    history = model.fit(
        x_train,
        y_train,
        epochs=10,
        batch_size=32,
        validation_data=(x_test, y_test),
        verbose=1
    )

    history_results[name] = history


# Plot Training and Validation Loss
plt.figure(figsize=(12, 5))

for name, history in history_results.items():
    plt.plot(
        history.history['loss'],
        label=name + " Training Loss"
    )

    plt.plot(
        history.history['val_loss'],
        label=name + " Validation Loss"
    )


plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Optimizer Comparison: Training and Validation Loss")
plt.legend()
plt.grid(True)

plt.show()

QUE : 4 -->

Use KerasTuner to automatically search for the best number of neurons in the first hidden layer for your MNIST classifier. Show the code you used and report the best configuration found.Hint: Use Hyperband or RandomSearch from keras_tuner and set a reasonable search space for neurons (e.g., 32 to 256).

In [ ]:
# Install KerasTuner if not installed:
# pip install keras-tuner

import tensorflow as tf
import keras_tuner as kt

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten


# Load MNIST dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Normalize data
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0


# Function to build model with hyperparameters
def build_model(hp):

    model = Sequential([
        Flatten(input_shape=(28, 28)),

        # Tune number of neurons
        Dense(
            units=hp.Int(
                'neurons',
                min_value=32,
                max_value=256,
                step=32
            ),
            activation='relu'
        ),

        Dense(10, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


# Create Hyperband tuner
tuner = kt.Hyperband(
    build_model,
    objective='val_accuracy',
    max_epochs=10,
    factor=3,
    directory='mnist_tuner',
    project_name='neuron_search'
)


# Search best model
tuner.search(
    x_train,
    y_train,
    epochs=10,
    validation_data=(x_test, y_test)
)


# Get best model
best_model = tuner.get_best_models(num_models=1)[0]


# Get best hyperparameters
best_hyperparameters = tuner.get_best_hyperparameters(num_trials=1)[0]

print("Best Number of Neurons:",
      best_hyperparameters.get('neurons'))


# Evaluate best model
test_loss, test_accuracy = best_model.evaluate(
    x_test,
    y_test
)

print("Best Model Test Accuracy:", test_accuracy)

QUE : 5 -->


Experiment with different weight initialization methods (e.g., 'he_normal', 'glorot_uniform', 'random_normal') in your Keras model and compare their impact on the model's training speed and accuracy.

In [ ]:
import tensorflow as tf
import time

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten


# Load MNIST dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Normalize data
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0


# Function to create model with different initializers
def create_model(initializer):

    model = Sequential([
        Flatten(input_shape=(28, 28)),

        Dense(
            128,
            activation='relu',
            kernel_initializer=initializer
        ),

        Dense(
            64,
            activation='relu',
            kernel_initializer=initializer
        ),

        Dense(
            10,
            activation='softmax'
        )
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


# Weight initialization methods
initializers = [
    'he_normal',
    'glorot_uniform',
    'random_normal'
]


results = []


# Train model with each initializer
for init in initializers:

    print("\nTraining with initializer:", init)

    model = create_model(init)

    start_time = time.time()

    history = model.fit(
        x_train,
        y_train,
        epochs=5,
        batch_size=32,
        validation_data=(x_test, y_test),
        verbose=1
    )

    end_time = time.time()

    training_time = end_time - start_time

    val_accuracy = history.history['val_accuracy'][-1]

    results.append(
        [init, training_time, val_accuracy]
    )


# Display results
print("\nInitializer | Training Time | Validation Accuracy")
print("-" * 55)

for result in results:
    print(
        result[0],
        "|",
        round(result[1], 2),
        "seconds |",
        round(result[2], 4)
    )